# COVID-19 Deaths Forecasting — JHU CSSE Format

## Provenance
- **Data:** JHU CSSE `time_series_covid19_deaths_US.csv`
- **Source:** https://github.com/CSSEGISandData/COVID-19
- **Generated by:** Claude (Anthropic) in conversation with Elizabeth Corey, March 2026
- **Purpose:** Review PyTorch LSTM + try HuggingFace Pipeline wrapping.  Working on JHU CSSE epidemiological data

## Key framing decisions
- JHU stores **cumulative** deaths — we convert to daily new deaths via diff()
- Daily counts have a **weekly reporting artifact** — smoothed with 7-day rolling average
- Forecast horizon (90 days) is **longer than look-back window** (60 days) — aggressive but
  defensible given COVID wave duration of roughly 60-90 days
- HuggingFace Pipeline wrapper is used **for interface convention**, not because HF provides
  a time-series model — the LSTM is pure PyTorch

## REVISIT IF
- Data source changes from cumulative to pre-differenced
- Forecast horizon or look-back window needs tuning for a different disease or geography
- Model is upgraded from LSTM to Transformer-based architecture (e.g. Temporal Fusion Transformer)


## 1. Installs

In [ ]:
!pip install -q transformers torch numpy pandas matplotlib scikit-learn

## 2. Imports & Config

In [ ]:
# ── IMPORTS & CONFIGURATION ───────────────────────────────────────────────────
#
# Standard ML stack: numpy/pandas for data, torch for the model,
# sklearn only for MinMaxScaler (no sklearn modeling is used),
# matplotlib for visualization, transformers for the Pipeline base class.
#
# CONTEXT: We import Pipeline from transformers but do NOT use any HuggingFace
#   model weights. The import is for the interface convention only — we subclass
#   Pipeline to make our LSTM behave like a standard HF pipeline.
# CONTEXT: This notebook was built and tested on PyTorch 2.x. The 'verbose'
#   argument was removed from ReduceLROnPlateau in PyTorch 2.x — do not add it.
# REVISIT IF: torch version < 2.0 (check with torch.__version__)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.preprocessing import MinMaxScaler
from transformers import Pipeline
import pickle
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')

# ── COLUMN INDICES (0-based) ──────────────────────────────────────────────────
# CONTEXT: JHU CSSE US deaths file has 12 metadata columns before the time series.
#   These indices were confirmed against the actual file header.
#   If the file structure changes, verify with: print(df.columns[:15])
# DECISION: indices set here once and referenced everywhere — do not hardcode
#   magic numbers elsewhere in the notebook.
COL_ADMIN2         = 5    # county name
COL_PROVINCE_STATE = 6    # state name
COL_COUNTRY_REGION = 7    # country name (always "US" in this file)
COL_SERIES_START   = 12   # first date column: "1/22/20" = Jan 22, 2020

# ── MODEL HYPERPARAMETERS ─────────────────────────────────────────────────────
# DECISION: SEQ_LEN=60, FORECAST_LEN=90
#   Forecast horizon is longer than look-back window. This is aggressive.
#   Defensible because COVID waves are roughly 60-90 day phenomena, so 60 days
#   of history should capture a wave's shape and momentum.
# TRADEOFF: A more conservative setup (SEQ_LEN=120, FORECAST_LEN=30) would
#   give the model more context but predict less far ahead.
# REVISIT IF: applying to a faster-moving signal (flu, RSV) where wave duration
#   is shorter — reduce SEQ_LEN and FORECAST_LEN proportionally.
SEQ_LEN      = 60    # look-back window in days
FORECAST_LEN = 90    # days ahead to predict

BATCH_SIZE   = 32    # standard for this data size; reduce if memory errors
EPOCHS       = 50    # with ReduceLROnPlateau, often converges before this

# DECISION: AdamW optimizer + ReduceLROnPlateau scheduler
#
# AdamW: Adam (Adaptive Moment Estimation) gives each parameter its own
#   learning rate, adjusted based on gradient history. Parameters with
#   consistent gradients get pushed harder; noisy ones get pushed cautiously.
#   The "W" fixes a bug in original Adam where weight decay interacted badly
#   with the gradient update — AdamW applies weight decay separately, which
#   acts as regularization and reduces overfitting on small datasets.
#   weight_decay=1e-4 is set in the optimizer call in the Training cell.
#
# ReduceLROnPlateau: watches validation loss and halves the learning rate
#   (factor=0.5) if it hasn't improved for patience=5 epochs. This is
#   reactive — it responds to actual training dynamics rather than a fixed
#   schedule. Early in training, larger steps move quickly through the loss
#   landscape; as the model converges, smaller steps prevent overshooting.
#
# TRADEOFF: ReduceLROnPlateau requires a validation set to watch. If val set
#   is too small (can happen with short location series), scheduler may fire
#   too early. Alternative: CosineAnnealingLR if val set is unreliable.
# REVISIT IF: training on a location with very few days of data — consider
#   switching to CosineAnnealingLR(optimizer, T_max=EPOCHS) instead.
LR = 1e-3  # initial learning rate; scheduler will reduce automatically


## 3. Load & Parse the JHU CSV

In [ ]:
# ── LOAD & PARSE THE JHU CSV ──────────────────────────────────────────────────
#
# CONTEXT: The JHU CSSE US deaths file is a wide-format CSV where each row is
#   a location and each column (after the 12 metadata columns) is a date.
#   This is the opposite of tidy/long format — do not attempt to melt() it
#   before extracting the time series or the column index logic breaks.
#
# CONTEXT: File is typically named time_series_covid19_deaths_US.csv
#   Source: https://github.com/CSSEGISandData/COVID-19/tree/master/csse_covid_19_data/csse_covid_19_time_series
#   The JHU dataset was frozen in March 2023 — it will not be updated further.
#
# DECISION: We assert column names at known indices rather than searching by
#   name. This makes the column index constants (set in Cell 2) the single
#   source of truth. If an assertion fails, fix the constants in Cell 2,
#   not here.
# REVISIT IF: a different JHU file is used (e.g. global deaths file) —
#   column indices will differ; reverify with print(df.columns[:15])

DATA_FILE = 'time_series_covid19_deaths_US.csv'   # <-- set your filename here

df = pd.read_csv(DATA_FILE, header=0, low_memory=False)

print(f'Shape: {df.shape}')
print(f'Total columns: {len(df.columns)}')
print(f'\nFirst 15 column names:')
for i, col in enumerate(df.columns[:15]):
    print(f'  [{i:2d}] {col}')
print(f'  ...')
print(f'  [{len(df.columns)-1:2d}] {df.columns[-1]}  (last column = last date)')

# ── VERIFY COLUMN STRUCTURE ───────────────────────────────────────────────────
# CONTEXT: These assertions are the canary. If any fail, the column indices
#   in Cell 2 are wrong for this file version. Fix constants there, not here.
# CONTEXT: '1/22/20' = January 22, 2020 — the first date JHU reported data.
#   This is a hardcoded string in the JHU file format, not derived from dates.
assert df.columns[COL_ADMIN2]         == 'Admin2',         \
    f'Col {COL_ADMIN2} is "{df.columns[COL_ADMIN2]}", expected "Admin2"'
assert df.columns[COL_PROVINCE_STATE] == 'Province_State', \
    f'Col {COL_PROVINCE_STATE} is "{df.columns[COL_PROVINCE_STATE]}", expected "Province_State"'
assert df.columns[COL_COUNTRY_REGION] == 'Country_Region', \
    f'Col {COL_COUNTRY_REGION} is "{df.columns[COL_COUNTRY_REGION]}", expected "Country_Region"'
assert df.columns[COL_SERIES_START]   == '1/22/20',        \
    f'Col {COL_SERIES_START} is "{df.columns[COL_SERIES_START]}", expected "1/22/20"'
print('\n✓ Column structure verified')

# ── PARSE DATE COLUMNS ────────────────────────────────────────────────────────
# CONTEXT: JHU date columns use M/DD/YY format with no zero-padding on month
#   (e.g. "1/22/20" not "01/22/20"). pandas format string '%m/%d/%y' handles
#   both padded and unpadded months correctly.
# CONTEXT: dates[] is a DatetimeIndex aligned 1:1 with the time series arrays
#   built in Cell 4. Index 0 = Jan 22 2020, index -1 = last day in the file.
#   This alignment is assumed throughout the notebook — do not slice date_cols
#   independently of the series arrays.
date_cols = df.columns[COL_SERIES_START:]
dates     = pd.to_datetime(date_cols, format='%m/%d/%y')
n_days    = len(date_cols)
print(f'\nDate range: {dates[0].date()} → {dates[-1].date()} ({n_days} days)')


## 4. Build Location Index & Time Series

In [ ]:
# ── BUILD LOCATION INDEX & TIME SERIES ───────────────────────────────────────
#
# DECISION: Convert cumulative → daily new deaths via np.diff()
#   JHU stores running totals, not daily counts. Example: if cumulative
#   deaths on day 5 = 10 and day 6 = 13, daily new deaths on day 6 = 3.
#   Feeding cumulative values to an LSTM trains it to learn only that
#   "tomorrow >= today" — monotonically true but useless for forecasting
#   future wave shape, peak timing, or decline rate.
#
# CONTEXT: np.diff(cumulative, prepend=0.0) prepends a virtual day-0 value
#   of 0 so the output array is the same length as the input. Without
#   prepend=0.0, diff() returns an array one element shorter, which would
#   break the 1:1 alignment with dates[] established in Cell 3.
#
# DECISION: Negative diffs are clipped to 0, not removed or interpolated.
#   CONTEXT: Negative values are real JHU data corrections — a county
#   revised its cumulative count downward (e.g. reclassified cause of death).
#   They are not sensor errors or parsing bugs. We clip rather than interpolate
#   because interpolation would invent deaths that didn't happen.
#   We warn rather than silently clip so the user knows which locations
#   had corrections and how many.
# TRADEOFF: Clipping to 0 understates deaths on correction days. For most
#   locations this is rare and small. For a few locations (e.g. New York City
#   in mid-2020) corrections were large and frequent — forecasts for those
#   locations will be less reliable.
# REVISIT IF: doing a rigorous epidemiological study rather than ML
#   demonstration — consider forward-filling or LOESS smoothing instead.
#
# CONTEXT: records[] is the central data structure for the rest of the
#   notebook. Each element is a dict with keys:
#     country, state, county  — location hierarchy strings
#     label                   — human-readable display name
#     cumulative              — int64 array, length n_days (raw JHU values)
#     daily                   — float32 array, length n_days (diff'd & clipped)
#     total_deaths            — int, cumulative[-1], used for sorting/browsing
#   All arrays are aligned with dates[] from Cell 3: index 0 = Jan 22 2020.

records = []
warnings_list = []

for idx, row in df.iterrows():
    country = str(row.iloc[COL_COUNTRY_REGION]).strip()
    state   = str(row.iloc[COL_PROVINCE_STATE]).strip()
    county  = str(row.iloc[COL_ADMIN2]).strip()

    # ── Extract cumulative series ─────────────────────────────────────────
    # CONTEXT: errors='coerce' turns any non-numeric cell (e.g. blank, "N/A")
    #   into NaN. fillna(0) treats missing reports as zero deaths — conservative
    #   but standard for JHU data where missing usually means no report filed,
    #   not that deaths are unknown.
    cumulative = pd.to_numeric(
        row.iloc[COL_SERIES_START:], errors='coerce'
    ).fillna(0).values.astype(np.float64)

    # ── Cumulative → daily new deaths ─────────────────────────────────────
    # DECISION: prepend=0.0 preserves array length = n_days
    #   so daily[] stays aligned with dates[]. See note above.
    daily = np.diff(cumulative, prepend=0.0)

    # ── Handle data corrections (negative diffs) ──────────────────────────
    n_negative = int((daily < 0).sum())
    if n_negative > 0:
        warnings_list.append(
            f'{country}/{state}/{county}: {n_negative} negative daily values '
            f'(JHU data corrections) — clipped to 0'
        )
        daily = np.clip(daily, 0, None)

    records.append({
        'country':      country,
        'state':        state,
        'county':       county,
        'label':        f'{county}, {state}' if county not in ('', 'nan') else state,
        'cumulative':   cumulative.astype(np.int64),
        'daily':        daily.astype(np.float32),
        'total_deaths': int(cumulative[-1])
    })

# ── INTEGRITY REPORT ──────────────────────────────────────────────────────────
# CONTEXT: This report is the first place to look if something seems wrong
#   downstream. High negative-diff counts in a location signal unreliable
#   data for that location — treat its forecast with extra skepticism.
print(f'Locations parsed: {len(records)}')
print(f'Locations with data corrections (negative diffs): {len(warnings_list)}')
if warnings_list:
    print('  First 10:')
    for w in warnings_list[:10]:
        print(f'    {w}')

# ── BUILD LOOKUP DATAFRAME ────────────────────────────────────────────────────
# CONTEXT: df_locs is used only for browsing and selection in Cell 5.
#   It is a lightweight index over records[] — it does not contain the
#   time series arrays. The source of truth for arrays is always records[].
df_locs = pd.DataFrame([{
    'country':      r['country'],
    'state':        r['state'],
    'county':       r['county'],
    'label':        r['label'],
    'total_deaths': r['total_deaths']
} for r in records])

print(f'\nTop 10 locations by total deaths:')
print(df_locs.nlargest(10, 'total_deaths')[['label', 'state', 'total_deaths']].to_string(index=False))


## 5. Pick a Location

In [ ]:
# ── PICK A LOCATION ───────────────────────────────────────────────────────────
#
# CONTEXT: This cell is interactive scaffolding — run the three sub-cells
#   sequentially to drill down from state → county → final selection.
#   The output of each sub-cell informs the input of the next.
#
# CONTEXT: Location hierarchy in this file is:
#   Country_Region (always "US") → Province_State → Admin2 (county)
#   Some rows have empty Admin2 — these are state-level aggregates,
#   not county-level rows. Their label will show just the state name.
#
# DECISION: We match on state + county strings exactly (case-sensitive).
#   If a match fails, the error message prints available counties for that
#   state so you can copy-paste the exact string rather than guess spelling.
# REVISIT IF: adding fuzzy matching for convenience — consider
#   difflib.get_close_matches(PICK_COUNTY, counties, n=3, cutoff=0.6)

# ── SUB-CELL 1: browse states ─────────────────────────────────────────────────
print('States / territories available:')
print(sorted(df_locs['state'].unique()))


In [ ]:
# ── SUB-CELL 2: browse counties within a state ────────────────────────────────
# CONTEXT: Set PICK_STATE to one of the strings printed by Sub-cell 1.
#   String must match exactly — copy-paste recommended.
PICK_STATE = 'California'    # <-- set state here
counties = sorted(df_locs[df_locs['state'] == PICK_STATE]['county'].unique())
print(f'Counties in {PICK_STATE}:')
print(counties)


In [ ]:
# ── SUB-CELL 3: final selection ───────────────────────────────────────────────
# CONTEXT: Set PICK_COUNTY to one of the strings printed by Sub-cell 2.
#   After this cell runs successfully, raw_deaths and cumul are the arrays
#   used by all downstream cells. Re-running this cell with a different
#   location and then re-running Cells 6-14 gives a complete forecast
#   for the new location without reloading the file.
# CONTEXT: raw_deaths is float32 daily new deaths (diff'd & clipped).
#   cumul is int64 cumulative deaths (raw JHU values, for reference only).
#   Both are aligned with dates[] from Cell 3: index 0 = Jan 22 2020.
PICK_COUNTY = 'Santa Clara'   # <-- set county here

matches = [r for r in records
           if r['state'] == PICK_STATE and r['county'] == PICK_COUNTY]

if len(matches) == 0:
    raise ValueError(
        f'No match: {PICK_STATE} / {PICK_COUNTY}\n'
        f'Available counties: {counties}'
    )
if len(matches) > 1:
    print(f'WARNING: {len(matches)} rows matched — using first')

selected   = matches[0]
raw_deaths = selected['daily']       # daily new deaths — used by all downstream cells
cumul      = selected['cumulative']  # cumulative totals — for reference/display only

print(f'Selected:      {selected["label"]}')
print(f'Date range:    {dates[0].date()} → {dates[-1].date()} ({n_days} days)')
print(f'Total deaths:  {int(cumul[-1]):,}')
print(f'Peak day:      {int(raw_deaths.max())} deaths on '
      f'{dates[int(raw_deaths.argmax())].date()}')
print(f'Days with >0:  {int((raw_deaths > 0).sum())}')

# ── PLOT ──────────────────────────────────────────────────────────────────────
# CONTEXT: Two-panel plot — raw daily counts (bar) + 7-day rolling average
#   (line). The weekly sawtooth pattern visible in the bars is the weekend
#   reporting artifact that motivates smoothing in Cell 6.
#   If the sawtooth is absent for this location, smoothing is still harmless.
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].bar(dates, raw_deaths, color='steelblue', alpha=0.6, width=1)
axes[0].set_title(f'Daily COVID Deaths — {selected["label"]}', fontsize=12)
axes[0].set_ylabel('Deaths / day')

rolling = pd.Series(raw_deaths).rolling(7, center=True, min_periods=1).mean()
axes[1].plot(dates, rolling, color='tomato', linewidth=1.5, label='7-day rolling avg')
axes[1].set_ylabel('7-day avg')
axes[1].legend()

for ax in axes:
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax.set_xlim(dates[0], dates[-1])

plt.tight_layout()
plt.show()


## 6. Preprocess — Smooth & Scale

In [ ]:
# ── PREPROCESS: SMOOTH & SCALE ────────────────────────────────────────────────
#
# This cell produces two arrays from raw_deaths:
#   smoothed  — noise-reduced daily deaths, same units as raw_deaths
#   scaled    — smoothed values mapped to [0, 1] for LSTM input
# Both are aligned with dates[] from Cell 3: index 0 = Jan 22 2020.
#
# ── STEP 1: 7-DAY ROLLING AVERAGE ────────────────────────────────────────────
# DECISION: Smooth raw_deaths with a centered 7-day rolling average.
#   CONTEXT: COVID death reporting had a strong weekly periodicity caused
#   by hospital and public health reporting cycles. Deaths were routinely
#   under-reported on weekends and over-reported on Mondays when batches
#   were filed. This is a reporting artifact, not a biological signal.
#   Without smoothing, the LSTM would partially learn this weekly sawtooth
#   pattern rather than the underlying epidemic wave dynamics.
#
#   center=True: the average is centered on each day, using 3 days before
#   and 3 days after. This preserves peak timing — a trailing average would
#   shift peaks rightward by ~3 days.
#
#   min_periods=1: allows the average at the edges of the series (first and
#   last 3 days) to be computed from fewer than 7 days rather than returning
#   NaN. Edge values are less reliable but NaN would break scaling.
#
# TRADEOFF: Smoothing reduces the sharpness of peak values. The LSTM will
#   tend to underestimate extreme single-day spikes. For most forecasting
#   purposes this is acceptable — we care more about wave shape and timing
#   than single-day extremes.
# TRADEOFF: center=True means the smoother uses future data relative to
#   each day. This is acceptable for training but would not be acceptable
#   in a true real-time deployment where future values are unknown.
# REVISIT IF: deploying in real-time — switch to trailing average
#   (center=False) or causal smoothing to avoid lookahead bias.
smoothed = pd.Series(raw_deaths).rolling(7, min_periods=1, center=True).mean().values.astype(np.float32)
smoothed = np.clip(smoothed, 0, None)  # defensive: rolling mean of clipped values
                                        # should never be negative, but clip anyway

# ── STEP 2: MINMAX SCALING ────────────────────────────────────────────────────
# DECISION: Scale smoothed values to [0, 1] using MinMaxScaler.
#   CONTEXT: LSTM gates use sigmoid and tanh activations internally.
#   These functions saturate (output ≈ 0 or ≈ 1) when inputs are large,
#   causing gradients to vanish and the model to stop learning. Raw death
#   counts ranging from 0 to 500+ would trigger this. Scaling to [0, 1]
#   keeps inputs in the activation functions' sensitive region.
#
#   CONTEXT: scaler is fit on the full series for the selected location,
#   then reused in the Pipeline (Cell 10) to scale inference inputs and
#   inverse-transform forecast outputs back to real death counts.
#   The scaler object must be saved alongside the model weights (Cell 14)
#   or inference will produce wrong-scaled outputs silently.
#
# TRADEOFF: MinMaxScaler is sensitive to outliers — a single extreme spike
#   compresses all other values into a narrow range. StandardScaler
#   (zero mean, unit variance) is more robust to outliers but produces
#   negative values which require care with the clip-to-zero logic.
# REVISIT IF: the selected location has extreme outlier spikes (visible
#   in the Cell 5 plot) — consider clipping raw_deaths at the 99th
#   percentile before scaling, or switching to StandardScaler.
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(smoothed.reshape(-1, 1)).flatten()

print(f'Smoothed — min: {smoothed.min():.2f}, max: {smoothed.max():.2f}, '
      f'mean: {smoothed.mean():.2f}')
print(f'Scaled   — min: {scaled.min():.4f}, max: {scaled.max():.4f}')


## 7. Dataset — Sliding Window

In [ ]:
# ── DATASET: SLIDING WINDOW ───────────────────────────────────────────────────
#
# DECISION: Frame the forecasting problem as supervised learning using a
#   sliding window over the scaled time series.
#   CONTEXT: An LSTM has no built-in notion of "give me the next 90 values."
#   We teach it by showing it many examples of the form:
#     input  = 60 consecutive days of scaled deaths  (the question)
#     target = the next 90 days of scaled deaths     (the answer)
#   The window slides one day at a time across the full series, generating
#   one training example per day. With ~800 training days, SEQ_LEN=60,
#   and FORECAST_LEN=90 this yields roughly 650 training windows.
#
# CONTEXT: TimeSeriesDataset returns tensors shaped:
#   x — (seq_len, 1)      the input window, unsqueezed to add feature dim
#   y — (forecast_len,)   the target sequence, flat
#   The feature dimension (the "1" in x) exists because LSTM expects
#   input_size >= 1. Here input_size=1 because we have one feature (deaths).
#   REVISIT IF: adding covariates (e.g. vaccination rates, mobility data)
#   — increase input_size and stack features in the last dimension of x.
#
# DECISION: 80/20 train/val split on time, not random.
#   CONTEXT: Random splitting would leak future information into training —
#   a window starting at day 900 could appear in training while day 850
#   appears in validation, meaning the model has "seen the future."
#   Time-ordered splitting ensures the model is always evaluated on data
#   it has never seen and that comes strictly after its training period.
# TRADEOFF: 80/20 gives ~880 train days and ~220 val days. With SEQ_LEN=60
#   and FORECAST_LEN=90, val windows require 150 days minimum — 220 days
#   yields only ~70 val windows, which is thin but workable.
# REVISIT IF: the val set is empty (notebook will warn) — this happens if
#   the selected location has a short series. Fix by reducing SEQ_LEN,
#   FORECAST_LEN, or the train fraction.
#
# CONTEXT: drop_last=True on train_loader discards the final incomplete
#   batch. This prevents BatchNorm instability (not used here but good
#   practice) and ensures all gradient steps use the full BATCH_SIZE.
#   drop_last=False on val_loader preserves all validation windows.

class TimeSeriesDataset(Dataset):
    def __init__(self, series, seq_len, forecast_len):
        self.series       = series
        self.seq_len      = seq_len
        self.forecast_len = forecast_len

    def __len__(self):
        # CONTEXT: max(0, ...) prevents negative length if the series is
        #   shorter than seq_len + forecast_len. The notebook warns about
        #   this condition after the split — do not remove the max(0).
        return max(0, len(self.series) - self.seq_len - self.forecast_len)

    def __getitem__(self, idx):
        x = self.series[idx : idx + self.seq_len]
        y = self.series[idx + self.seq_len : idx + self.seq_len + self.forecast_len]
        return (
            torch.tensor(x, dtype=torch.float32).unsqueeze(-1),  # (seq_len, 1)
            torch.tensor(y, dtype=torch.float32)                  # (forecast_len,)
        )

# ── TRAIN / VAL SPLIT ─────────────────────────────────────────────────────────
# CONTEXT: split is an integer index into scaled[], not a date.
#   split_date is derived from dates[] for display only — it is not used
#   in any computation. The alignment between split and split_date depends
#   on dates[] and scaled[] being the same length, which is guaranteed by
#   Cell 6 (smoothed and scaled preserve array length).
split      = int(len(scaled) * 0.8)
split_date = dates[split]
train_ds   = TimeSeriesDataset(scaled[:split], SEQ_LEN, FORECAST_LEN)
val_ds     = TimeSeriesDataset(scaled[split:], SEQ_LEN, FORECAST_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f'Train: days 0–{split} (up to {split_date.date()}) → {len(train_ds)} windows')
print(f'Val:   days {split}–{n_days} → {len(val_ds)} windows')

# ── INTEGRITY CHECK ───────────────────────────────────────────────────────────
# CONTEXT: Empty val set means the location's post-split series is too short
#   for even one window of SEQ_LEN + FORECAST_LEN days. The model will still
#   train but you will get no validation metrics and no early stopping signal.
#   The forecast in Cell 11 will still run — it uses the full series, not
#   the val set — but treat results with extra skepticism.
if len(val_ds) == 0:
    print('WARNING: Val set empty. Reduce SEQ_LEN or FORECAST_LEN, '
          'or choose a location with more data.')


## 8. LSTM Model

In [ ]:
# ── LSTM MODEL ────────────────────────────────────────────────────────────────
#
# DECISION: A 2-layer stacked LSTM with a linear output head.
#   CONTEXT: LSTM (Long Short-Term Memory) is a recurrent architecture
#   designed to learn dependencies across long sequences. It solves the
#   "vanishing gradient" problem of vanilla RNNs by using three gating
#   mechanisms:
#     forget gate  — decides what to discard from memory
#     input gate   — decides what new information to store
#     output gate  — decides what to expose to the next layer
#   These gates allow the model to selectively remember patterns across
#   60-day windows — e.g. the shape of a rising wave — while forgetting
#   irrelevant noise.
#
# DECISION: hidden_size=128
#   CONTEXT: Sized for a single-feature time series (~1100 days) with
#   complex multi-wave dynamics. Larger than the sine-wave prototype
#   (hidden_size=64) because real epidemiological data has more structure:
#   multiple overlapping waves, policy interventions, vaccination effects.
#   Not rigorously tuned — chosen by rule of thumb.
# TRADEOFF: Larger hidden_size increases capacity but also overfitting risk
#   on small datasets. Dropout (below) partially mitigates this.
# REVISIT IF: validation loss is much higher than training loss (overfit) —
#   reduce hidden_size to 64 or increase dropout toward 0.3-0.4.
# REVISIT IF: validation loss is still high after many epochs (underfit) —
#   increase hidden_size to 256 or add a third LSTM layer.
#
# DECISION: num_layers=2 (stacked LSTM)
#   CONTEXT: The first LSTM layer learns low-level temporal patterns
#   (weekly cycles, short-term trends). The second layer learns higher-level
#   patterns across the first layer's output (wave shape, surge onset).
#   A single layer would miss this hierarchical structure; three or more
#   layers would be hard to train on this data volume.
# CONTEXT: dropout is applied between LSTM layers, not after the final
#   layer. PyTorch's nn.LSTM dropout argument only acts between layers —
#   this is why we apply nn.Dropout explicitly after the LSTM output.
#   If num_layers=1, PyTorch ignores the dropout argument entirely —
#   the code sets dropout=0.0 in that case to be explicit.
#
# DECISION: dropout=0.2
#   CONTEXT: Randomly zeros 20% of neurons during each training step,
#   forcing the network to learn redundant representations and reducing
#   reliance on any single neuron. Standard starting point for this
#   data size. Applied both between LSTM layers and before the linear head.
# TRADEOFF: dropout=0.0 at inference time (model.eval() disables it).
#   This means the model behaves differently during training vs inference
#   — this is intentional and correct, not a bug.
#
# DECISION: Linear output head maps hidden_size → forecast_len in one step.
#   CONTEXT: The LSTM processes the full 60-day input sequence but we only
#   use the final timestep's hidden state (out[:, -1]) as input to the
#   linear head. This is the "many-to-one then project" pattern — the LSTM
#   summarizes the entire sequence into a single hidden vector, and the
#   linear layer expands that vector into 90 forecast values simultaneously.
# TRADEOFF: Predicting all 90 days at once (direct multi-step forecasting)
#   is less accurate than recursive forecasting (predict day+1, feed back,
#   predict day+2, etc.) for short horizons, but avoids error accumulation
#   over the 90-day forecast window. For COVID wave forecasting, direct
#   multi-step is the better tradeoff.
# REVISIT IF: forecast quality degrades sharply after ~30 days — consider
#   a seq2seq architecture with an encoder LSTM and a decoder LSTM that
#   generates outputs one step at a time with attention.
#
# NOTE TO SELF: ARCHITECTURE SCALING
# CONTEXT: This model is trained on a single location (~1100 days, 1 feature).
#   At this data volume, hidden_size=128 and num_layers=2 are at the upper
#   limit of what is justifiable without overfitting.
# REVISIT IF: combining multiple locations into a single training set
#   (e.g. all California counties, or all US counties) — the much larger
#   dataset would justify num_layers=3 or 4 and hidden_size=256, and would
#   allow the model to learn dynamics that generalize across locations rather
#   than fitting a single location's idiosyncrasies.
#
# NOTE TO SELF: NONLINEAR UNDERLYING DYNAMICS
# CONTEXT: COVID wave dynamics are governed by nonlinear epidemiological
#   processes approximated by SIR-type models (Susceptible-Infected-Recovered)
#   — systems of nonlinear differential equations. The LSTM approximates
#   these dynamics implicitly from data rather than modeling them explicitly.
#   Between waves, the parameters driving the dynamics shifted (transmissibility,
#   immune escape, severity) in ways the LSTM cannot observe directly —
#   it sees only the resulting death time series.
#   This means:
#   1. The model can only approximate dynamics it has seen during training.
#      Novel variant waves may fall outside the training distribution entirely.
#   2. The model has no causal structure — it cannot reason about
#      interventions (lockdowns, vaccines) it hasn't seen correlated
#      with outcome changes in training data.
# REVISIT IF: forecast quality is poor at wave onset — consider augmenting
#   input features with leading indicators (mobility data, wastewater
#   surveillance) that correlate with case growth before death growth.
#
# NOTE TO SELF: LSTM IS NOT A FREQUENCY DECOMPOSER
# CONTEXT: LSTMs learn temporal patterns (shape, momentum, sequence) through
#   gating, not frequency components. This distinction matters here because:
#   1. Frequency decomposition (FFT, wavelets) assumes stationarity —
#      that frequency content is constant over time. COVID waves are
#      non-stationary: the parameters driving wave dynamics shifted between
#      variants in ways that changed wave shape, speed, and amplitude.
#   2. The LSTM's hidden state carries temporal context in order, allowing
#      it to adapt to non-stationarity in principle — but only for
#      regime shifts similar to those seen during training.
#   3. The weekly reporting artifact (smoothed in Cell 6) is the closest
#      thing to a true frequency component in this signal — and we removed
#      it deliberately before the LSTM sees the data.
# REVISIT IF: exploring explicit frequency decomposition as a complement —
#   STL decomposition (statsmodels) can separate trend, seasonal, and
#   residual components before feeding to the LSTM (a hybrid approach).

class LSTMForecaster(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=2,
                 dropout=0.2, forecast_len=90):
        super().__init__()

        # CONTEXT: batch_first=True means input tensor shape is
        #   (batch, seq_len, input_size) rather than PyTorch's default
        #   (seq_len, batch, input_size). batch_first=True matches the
        #   shape produced by TimeSeriesDataset and is easier to reason about.
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        # CONTEXT: applied after LSTM, before linear head.
        #   Disabled automatically during model.eval() — no manual toggle needed.
        self.dropout = nn.Dropout(dropout)

        # CONTEXT: maps final hidden state → all forecast_len outputs at once.
        #   See "direct multi-step forecasting" note above.
        self.fc = nn.Linear(hidden_size, forecast_len)

    def forward(self, x):
        # x shape:        (batch, seq_len, input_size)
        out, _ = self.lstm(x)
        # out shape:      (batch, seq_len, hidden_size)
        # out[:, -1]:     (batch, hidden_size) — final timestep only
        # CONTEXT: _ contains (h_n, c_n) — the final hidden and cell states.
        #   Discarded here because we use the full output sequence's last
        #   timestep rather than h_n directly. They carry the same information
        #   for a single forward pass but out[:, -1] is more explicit.
        return self.fc(self.dropout(out[:, -1]))
        # return shape:   (batch, forecast_len)

model = LSTMForecaster(
    input_size=1,
    hidden_size=128,
    num_layers=2,
    dropout=0.2,
    forecast_len=FORECAST_LEN
).to(device)

print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')


## 9. Training

In [ ]:
# ── TRAINING ──────────────────────────────────────────────────────────────────
#
# CONTEXT: The training loop follows the standard PyTorch pattern:
#   zero gradients → forward pass → compute loss → backward pass → step.
#   Each epoch processes the full training set in batches, then evaluates
#   on the full validation set with gradients disabled.
#
# DECISION: MSELoss (Mean Squared Error) as the loss function.
#   CONTEXT: MSE penalizes large errors more than small ones (squared term).
#   For death count forecasting this is desirable — underestimating a peak
#   by 50 deaths is much worse than underestimating a trough by 5 deaths.
#   MSE naturally encodes this asymmetric cost without explicit weighting.
# TRADEOFF: MSE is sensitive to outliers — a single anomalous spike in the
#   training data can dominate the loss and distort training. Smoothing in
#   Cell 6 partially mitigates this. Alternative: MAE (L1) loss is more
#   robust to outliers but treats all errors equally regardless of magnitude.
# REVISIT IF: training is unstable or dominated by a few extreme days —
#   consider Huber loss (nn.SmoothL1Loss) which behaves like MSE for small
#   errors and MAE for large ones.
#
# DECISION: Gradient clipping at norm=1.0
#   CONTEXT: LSTMs are prone to exploding gradients — a single bad batch
#   can produce very large gradients that cause the optimizer to take a
#   catastrophically large step, destabilizing training. Clipping rescales
#   the gradient vector if its norm exceeds 1.0, preserving direction but
#   bounding magnitude.
#   This is standard practice for RNN/LSTM training and nearly always
#   harmless — if gradients are well-behaved, clipping never fires.
# REVISIT IF: loss is not decreasing at all in early epochs — clipping
#   threshold may be too aggressive; try increasing to 5.0.
#
# DECISION: Save best model by validation loss, reload at end.
#   CONTEXT: ReduceLROnPlateau may allow training loss to keep decreasing
#   after validation loss has stopped improving (overfitting). Saving the
#   best validation checkpoint and reloading it at the end ensures we use
#   the model that generalizes best, not the most recently trained one.
#   File: 'best_covid_model.pt' — overwritten each time this cell runs.
# REVISIT IF: running multiple experiments — add a timestamp or location
#   name to the filename to avoid overwriting across runs.
#
# NOTE TO SELF: training on a single location means the model is fitting
#   one time series of ~880 training days. This is a small dataset by ML
#   standards. If training loss drops quickly but validation loss plateaus
#   high, the model has memorized the training waves and is not generalizing.
#   The primary remedy is more data — see multi-location note in Cell 8.

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=5, factor=0.5
)

train_losses, val_losses = [], []
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    # ── TRAIN ─────────────────────────────────────────────────────────────
    model.train()
    # CONTEXT: model.train() enables dropout. model.eval() below disables it.
    #   This is not a bug — dropout is intentionally active during training
    #   and inactive during evaluation. See Cell 8 dropout note.
    batch_losses = []
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        # CONTEXT: zero_grad() must be called before each backward pass.
        #   Without it, gradients accumulate across batches — a common
        #   PyTorch gotcha that produces incorrect updates silently.
        loss = criterion(model(x), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        batch_losses.append(loss.item())

    # ── VALIDATE ──────────────────────────────────────────────────────────
    model.eval()
    val_batch_losses = []
    with torch.no_grad():
        # CONTEXT: torch.no_grad() disables gradient computation entirely.
        #   Required during validation — we are not updating weights here,
        #   and computing gradients would waste memory and time.
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            val_batch_losses.append(criterion(model(x), y).item())

    train_loss = np.mean(batch_losses)
    val_loss   = np.mean(val_batch_losses) if val_batch_losses else float('nan')
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    # ── CHECKPOINT ────────────────────────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_covid_model.pt')

    if (epoch + 1) % 10 == 0:
        cur_lr = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | '
              f'train={train_loss:.5f} | '
              f'val={val_loss:.5f} | '
              f'lr={cur_lr:.2e}')

# ── RELOAD BEST CHECKPOINT ────────────────────────────────────────────────────
# CONTEXT: Reloads the weights saved at the epoch with lowest val loss.
#   From this point forward, model is in eval() mode with best weights.
#   All downstream cells (10-14) use this model state.
model.load_state_dict(torch.load('best_covid_model.pt', map_location=device))
print(f'\nBest val loss: {best_val_loss:.5f}')

plt.figure(figsize=(10, 3))
plt.plot(train_losses, label='Train MSE')
plt.plot(val_losses,   label='Val MSE')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Training Curve')
# CONTEXT: A well-behaved training run shows both curves decreasing together,
#   with val loss slightly above train loss. Watch for:
#   - val loss increasing while train loss decreases → overfitting
#   - both losses plateauing high → underfitting or lr too low
#   - loss spikes → exploding gradients (increase clipping or reduce lr)
plt.legend()
plt.tight_layout()
plt.show()


## 10. HuggingFace Pipeline Wrapper

In [ ]:
# ── HUGGINGFACE PIPELINE WRAPPER ──────────────────────────────────────────────
#
# DECISION: Wrap the LSTM in a HuggingFace Pipeline subclass.
#   CONTEXT: HuggingFace's Pipeline base class is the standard interface
#   for deploying models in the HF ecosystem. By conforming to this interface
#   our LSTM becomes interchangeable with any HF model from the outside —
#   the caller does not need to know it is a PyTorch LSTM rather than a
#   Transformer. This is the value of the wrapper: not functionality, but
#   interface conformance and extensibility.
#
# CONTEXT: HuggingFace Pipeline normally assumes an NLP tokenizer exists.
#   For non-NLP models we bypass Pipeline.__init__ entirely and wire the
#   three lifecycle methods manually:
#     preprocess()       — raw input → model-ready tensor
#     _forward()         — tensor → raw model output
#     postprocess()      — raw output → human-readable result
#   __call__() chains these three in order. This is the standard pattern
#   for non-NLP HF pipelines — it is intentional, not a hack.
#
# DECISION: preprocess() applies the same smoothing and scaling as Cell 6.
#   CONTEXT: Inference inputs must be transformed identically to training
#   inputs or the model receives out-of-distribution data and produces
#   wrong outputs silently. The pipeline encapsulates this transformation
#   so callers cannot accidentally skip it.
#   CONTEXT: scaler was fit in Cell 6 on the selected location's full series.
#   It must be the same scaler object used here — a new scaler fit on a
#   different series would produce different scaling and wrong forecasts.
#   This is why scaler is saved alongside model weights in Cell 14.
#
# DECISION: preprocess() takes the last seq_len days of the input
#   automatically, regardless of how long the input array is.
#   CONTEXT: This allows the caller to pass the full raw_deaths array
#   without manually slicing — the pipeline finds the right window.
#   REVISIT IF: you want to forecast from a specific historical window
#   rather than the most recent — add a start_idx parameter to preprocess().
#
# DECISION: postprocess() clips forecast output to >= 0.
#   CONTEXT: The linear output head has no non-negativity constraint —
#   it can produce negative values, especially in the tail of the forecast
#   where the model is least confident. Negative deaths are physically
#   impossible. We clip rather than use a softplus activation because
#   clipping is transparent and reversible; softplus would change the
#   model architecture and require retraining.
#
# CONTEXT: last_known_date is stored in the pipeline to generate forecast
#   date strings in postprocess(). It must match the last date in dates[]
#   from Cell 3. If you reload this pipeline from saved weights (Cell 14)
#   you must pass the correct last_known_date explicitly — it is not
#   stored in the model weights file.
# REVISIT IF: operationalizing this pipeline with fresh data — last_known_date
#   should be updated to the last date in the new data, not hardcoded.

class COVIDForecastPipeline(Pipeline):
    def __init__(self, model, scaler, seq_len, last_known_date, **kwargs):
        # CONTEXT: We do not call super().__init__() because the HF parent
        #   requires a tokenizer we don't have. All required state is set
        #   explicitly below. This is safe — Pipeline.__init__ only sets
        #   attributes we don't need for non-NLP use.
        self.model           = model
        self.scaler          = scaler
        self.seq_len         = seq_len
        self.last_known_date = pd.Timestamp(last_known_date)
        self._device         = next(model.parameters()).device
        self.model.eval()
        # CONTEXT: model.eval() here is defensive — training cell reloads
        #   best weights and leaves model in eval() already, but we set it
        #   again here so the pipeline is safe to instantiate independently
        #   (e.g. after reloading from Cell 14 save/reload template).

    def _sanitize_parameters(self, **kwargs):
        # CONTEXT: Required by Pipeline interface. Returns three empty dicts
        #   because we have no per-call parameters to parse. If you add
        #   parameters (e.g. smoothing window, forecast horizon) they would
        #   be parsed and routed here to preprocess/forward/postprocess kwargs.
        return {}, {}, {}

    def preprocess(self, inputs):
        # CONTEXT: inputs is raw (unsmoothed, unscaled) daily death counts.
        #   Must be a list or np.array of length >= seq_len.
        #   Applies identical smoothing to Cell 6 — see that cell for
        #   rationale on center=True and the weekend reporting artifact.
        arr      = np.array(inputs, dtype=np.float32)
        smoothed = pd.Series(arr).rolling(7, min_periods=1, center=True).mean().values
        smoothed = np.clip(smoothed, 0, None)

        # Take last seq_len days — the most recent context window
        window = smoothed[-self.seq_len:]
        scaled = self.scaler.transform(window.reshape(-1, 1)).flatten()

        # Shape: (1, seq_len, 1) — batch=1, seq_len days, input_size=1 feature
        tensor = torch.tensor(scaled, dtype=torch.float32).unsqueeze(0).unsqueeze(-1)
        return {'input_tensor': tensor.to(self._device)}

    def _forward(self, model_inputs):
        with torch.no_grad():
            # CONTEXT: torch.no_grad() is essential here — we are not training.
            #   output shape: (1, forecast_len)
            output = self.model(model_inputs['input_tensor'])
        return {'output_tensor': output}

    def postprocess(self, model_outputs):
        out = model_outputs['output_tensor'].cpu().numpy()

        # Inverse-scale: mapped [0,1] back to deaths/day units
        # CONTEXT: scaler.inverse_transform requires shape (-1, 1),
        #   hence the reshape. flatten() restores to 1D for output.
        out_unscaled = np.clip(
            self.scaler.inverse_transform(out.reshape(-1, 1)).flatten(),
            0, None   # clip negative values — see DECISION note above
        )

        # Generate forecast date strings aligned to last_known_date
        # CONTEXT: day i+1 because day 0 is last_known_date itself (already known).
        forecast_dates = [
            (self.last_known_date + pd.Timedelta(days=i+1)).strftime('%Y-%m-%d')
            for i in range(len(out_unscaled))
        ]
        return {
            'forecast': out_unscaled.tolist(),
            'dates':    forecast_dates,
            'steps':    len(out_unscaled)
        }

    def __call__(self, inputs):
        # CONTEXT: chains preprocess → _forward → postprocess in order.
        #   This mirrors how HF Pipeline.__call__ works internally but is
        #   explicit here so the data flow is visible without reading HF source.
        return self.postprocess(self._forward(self.preprocess(inputs)))


# ── INSTANTIATE ───────────────────────────────────────────────────────────────
# CONTEXT: dates[-1] is the last date in the JHU series — the anchor for
#   all forecast date generation. If reloading from saved weights (Cell 14),
#   pass the same value here explicitly.
forecast_pipeline = COVIDForecastPipeline(
    model=model,
    scaler=scaler,
    seq_len=SEQ_LEN,
    last_known_date=dates[-1]
)
print('Pipeline ready.')


## 11. Forecast the Next 90 Days

In [ ]:
# ── FORECAST THE NEXT 90 DAYS ─────────────────────────────────────────────────
#
# CONTEXT: This cell runs inference via the pipeline built in Cell 10.
#   It feeds the full raw_deaths array — the pipeline's preprocess() method
#   automatically takes the last seq_len=60 days as the context window.
#   Output is a dict with three keys:
#     'forecast' — list of float, length=90, in deaths/day units
#     'dates'    — list of date strings, length=90, starting day after
#                  the last known date in the JHU series
#     'steps'    — int, always 90 for this configuration
#
# CONTEXT: The forecast extends beyond the JHU dataset end date (March 2023).
#   There is no ground truth to compare against — this is a true out-of-sample
#   forecast. Validation metrics from Cell 13 are the best available proxy
#   for expected forecast accuracy.
#
# DECISION: We forecast from the end of the full series, not the end of the
#   training set. The model was trained on 80% of the data but we use 100%
#   as the context window for the final forecast.
#   CONTEXT: This is standard practice — after training and validation we
#   use all available data to inform the final forecast. Withholding the
#   last 20% from the context window would artificially degrade forecast
#   quality for no benefit at inference time.
# TRADEOFF: The model has never been trained on the last 20% of the data
#   as input — it has only seen those days as validation targets. The context
#   window therefore contains some days the model has not learned from as
#   inputs. In practice this is a minor concern for a 60-day window ending
#   in March 2023 — the dynamics in late 2022/early 2023 are within the
#   range of patterns seen during training.
#
# NOTE TO SELF: forecast quality degrades with horizon length. The model's
#   90-day forecast should be interpreted as:
#   - Days 1-30:  reasonable estimate of near-term trajectory
#   - Days 31-60: directional indicator, wide uncertainty
#   - Days 61-90: speculative — treat as scenario rather than prediction
#   The model has no mechanism to express uncertainty (no confidence
#   intervals). REVISIT IF: uncertainty quantification is needed — consider
#   Monte Carlo dropout (run inference N times with model.train() active,
#   take mean and std across runs) or a probabilistic output head.

result = forecast_pipeline(raw_deaths)

forecast_dates  = pd.to_datetime(result['dates'])
forecast_values = np.array(result['forecast'])

print(f'Forecast period: {result["dates"][0]} → {result["dates"][-1]}')
print(f'Peak forecast:   {forecast_values.max():.1f} deaths/day '
      f'on {result["dates"][int(forecast_values.argmax())]}')
print(f'Total forecast:  {forecast_values.sum():.0f} deaths over 90 days')

# ── HORIZON BREAKDOWN ─────────────────────────────────────────────────────────
# CONTEXT: Splits the forecast into three reliability bands per the note above.
#   These are interpretive guidelines, not hard boundaries.
print(f'\nHorizon reliability breakdown:')
print(f'  Days  1-30 (reasonable):    '
      f'avg {forecast_values[:30].mean():.1f} deaths/day')
print(f'  Days 31-60 (directional):   '
      f'avg {forecast_values[30:60].mean():.1f} deaths/day')
print(f'  Days 61-90 (speculative):   '
      f'avg {forecast_values[60:].mean():.1f} deaths/day')


## 12. Visualize

In [ ]:
# ── VISUALIZE ─────────────────────────────────────────────────────────────────
#
# CONTEXT: Three-layer plot showing history and forecast together:
#   1. Raw daily deaths (bar)       — shows true data including reporting artifacts
#   2. 7-day rolling average (line) — shows smoothed signal the model was trained on
#   3. LSTM forecast (line+markers) — 90-day forecast from pipeline
#
# DECISION: Show HISTORY_DAYS=180 of history alongside the forecast.
#   CONTEXT: 180 days (6 months) is enough to show the most recent wave
#   shape and the decay into the forecast period, without compressing the
#   forecast region into a small fraction of the x-axis.
#   The full series plot is in Cell 5 — this plot is for forecast context.
# REVISIT IF: the selected location had a major wave in the 180 days before
#   the forecast — consider increasing to 270 or 365 to show the full wave.
# REVISIT IF: generating plots for a report — increase figsize and dpi,
#   and consider removing the raw bar chart for cleaner presentation.
#
# CONTEXT: The shaded forecast region (axvspan) and the vertical dashed line
#   (axvline) together mark the boundary between known data and forecast.
#   This visual boundary is important — without it the forecast line blends
#   into the history line and the distinction is easy to miss.
#
# NOTE TO SELF: The forecast line will appear smooth relative to the raw
#   history bars. This is expected — the model was trained on smoothed data
#   (Cell 6) and its output is inherently smooth. It is not evidence that
#   the model is overconfident or underfit.

HISTORY_DAYS = 180

hist_dates  = dates[-HISTORY_DAYS:]
hist_raw    = raw_deaths[-HISTORY_DAYS:]
hist_smooth = pd.Series(raw_deaths).rolling(
    7, min_periods=1, center=True
).mean().values[-HISTORY_DAYS:]

fig, ax = plt.subplots(figsize=(15, 5))

# ── LAYER 1: raw daily deaths ─────────────────────────────────────────────────
# CONTEXT: alpha=0.3 makes bars recede visually so the smooth lines
#   are readable on top. width=1 fills gaps between daily bars.
ax.bar(hist_dates, hist_raw,
       color='steelblue', alpha=0.3, width=1,
       label='Raw daily deaths')

# ── LAYER 2: 7-day rolling average ───────────────────────────────────────────
# CONTEXT: This is the signal the model actually learned from — showing it
#   here makes the forecast visually continuous with its training input.
ax.plot(hist_dates, hist_smooth,
        color='steelblue', linewidth=1.5,
        label='7-day avg (history)')

# ── LAYER 3: LSTM forecast ────────────────────────────────────────────────────
# CONTEXT: markersize=2 adds subtle dots at each forecast day without
#   cluttering the line. Useful for reading individual day values on hover
#   in interactive backends (e.g. %matplotlib widget in JupyterLab).
ax.plot(forecast_dates, forecast_values,
        color='tomato', linewidth=2,
        marker='o', markersize=2,
        label='LSTM 90-day forecast')

# ── BOUNDARY MARKERS ──────────────────────────────────────────────────────────
# CONTEXT: axvspan shades the forecast region. axvline marks the exact
#   boundary day. Together they make the known/forecast transition
#   unambiguous even in grayscale printouts.
ax.axvspan(forecast_dates[0], forecast_dates[-1],
           alpha=0.07, color='tomato')
ax.axvline(x=dates[-1],
           color='gray', linestyle='--', linewidth=1,
           label='Last known data')

# ── HORIZON RELIABILITY BANDS ─────────────────────────────────────────────────
# CONTEXT: Visual encoding of the three reliability bands from Cell 11.
#   Lighter shading = less reliable. These are interpretive guidelines.
#   Comment out this block if the shading is visually distracting.
band_colors = [(0.05, 'tomato'), (0.08, 'orange'), (0.12, 'gray')]
band_boundaries = [
    (forecast_dates[0],  forecast_dates[29]),   # days  1-30  reasonable
    (forecast_dates[30], forecast_dates[59]),   # days 31-60  directional
    (forecast_dates[60], forecast_dates[89]),   # days 61-90  speculative
]
band_labels = ['days 1-30 (reasonable)',
               'days 31-60 (directional)',
               'days 61-90 (speculative)']

for (alpha, color), (start, end), label in zip(band_colors, band_boundaries, band_labels):
    ax.axvspan(start, end, alpha=alpha, color=color, label=label)

# ── FORMATTING ────────────────────────────────────────────────────────────────
ax.set_title(
    f'COVID Deaths Forecast — {selected["label"]}\n'
    f'Last {HISTORY_DAYS} days history + 90-day LSTM forecast '
    f'(shading = reliability bands)',
    fontsize=12
)
ax.set_ylabel('Deaths per day')
ax.set_ylim(bottom=0)
# CONTEXT: interval=1 shows every month on x-axis. If the plot is too
#   crowded, increase to interval=2 or interval=3.
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()


## 13. Evaluate on Validation Set

In [ ]:
# ── EVALUATE ON VALIDATION SET ───────────────────────────────────────────────
#
# CONTEXT: This cell measures how well the model predicts on data it was
#   not trained on — the last 20% of the series (roughly late 2022 to
#   March 2023 depending on location). These metrics are the best available
#   proxy for expected forecast accuracy on the true 90-day forecast in
#   Cell 11, since that forecast has no ground truth to compare against.
#
# DECISION: Evaluate in original deaths/day units, not scaled units.
#   CONTEXT: Scaled MSE (what the model optimizes during training) is
#   dimensionless and uninterpretable — "0.003 MSE" means nothing to a
#   reader. Inverse-scaling back to deaths/day gives MAE and RMSE values
#   that are directly interpretable: "the model is off by X deaths/day
#   on average." This is the only place in the notebook where we care
#   about human-interpretable error magnitude rather than optimizer signal.
#
# DECISION: Report both MAE and RMSE.
#   CONTEXT: MAE (Mean Absolute Error) — average absolute deviation in
#   deaths/day. Treats all errors equally regardless of magnitude.
#   Intuitive: "on average the model is off by X deaths/day."
#
#   RMSE (Root Mean Squared Error) — square root of average squared error.
#   Penalizes large errors more than small ones (same reasoning as MSELoss
#   in Cell 9). If RMSE >> MAE, the model is making occasional large errors
#   rather than consistent small ones — often happens at wave peaks.
#
# CONTEXT: We evaluate only on 1-day-ahead predictions (preds_raw[:, 0]
#   vs actuals_raw[:, 0]) in the validation plot. This is the model's
#   most accurate prediction horizon and gives the clearest signal about
#   whether it has learned the underlying dynamics. The full 90-day
#   forecast metrics would be dominated by compounding uncertainty and
#   would be harder to interpret visually.
#
# NOTE TO SELF: Interpret validation metrics in the context of the
#   location's death counts. An MAE of 5 deaths/day means something very
#   different for Santa Clara County (peak ~30/day) vs Los Angeles County
#   (peak ~300/day). Consider normalizing by peak deaths for cross-location
#   comparison:
#     normalized_mae = mae / raw_deaths.max()
# REVISIT IF: comparing model performance across multiple locations —
#   add normalized MAE to the output so comparisons are meaningful.
#
# NOTE TO SELF: If val set is empty (warned in Cell 7), this cell will
#   print a warning and produce no metrics or plot. The forecast in Cell 11
#   will still have run — but without validation metrics there is no
#   quantitative basis for trusting it.

model.eval()
all_preds, all_actuals = [], []

with torch.no_grad():
    # CONTEXT: iterating val_loader gives windows from the validation
    #   portion of the series only (days split: onward). Each window
    #   produces a (batch, forecast_len) prediction tensor.
    for x, y in val_loader:
        x = x.to(device)
        all_preds.append(model(x).cpu().numpy())
        all_actuals.append(y.numpy())

if all_preds:
    all_preds   = np.concatenate(all_preds,   axis=0)  # (N, forecast_len)
    all_actuals = np.concatenate(all_actuals, axis=0)  # (N, forecast_len)

    # ── INVERSE SCALE ─────────────────────────────────────────────────────
    # CONTEXT: inv() restores scaled [0,1] values to deaths/day units using
    #   the same scaler fit in Cell 6. clip(0) removes physically impossible
    #   negative predictions — see Cell 10 postprocess() note for rationale.
    def inv(arr):
        return np.clip(
            scaler.inverse_transform(arr.reshape(-1, 1)).reshape(arr.shape),
            0, None
        )

    preds_raw   = inv(all_preds)
    actuals_raw = inv(all_actuals)

    # ── METRICS ───────────────────────────────────────────────────────────
    mae  = np.mean(np.abs(preds_raw - actuals_raw))
    rmse = np.sqrt(np.mean((preds_raw - actuals_raw) ** 2))
    normalized_mae = mae / raw_deaths.max() if raw_deaths.max() > 0 else float('nan')

    print(f'Validation metrics (deaths/day units):')
    print(f'  MAE:           {mae:.2f} deaths/day')
    print(f'  RMSE:          {rmse:.2f} deaths/day')
    print(f'  Normalized MAE:{normalized_mae:.3f} (fraction of peak daily deaths)')
    print(f'  Note: RMSE >> MAE suggests occasional large errors at wave peaks')
    print(f'        RMSE ≈ MAE suggests consistent errors across the series')

    # ── VALIDATION PLOT ───────────────────────────────────────────────────
    # CONTEXT: Plots 1-day-ahead predicted vs actual across the full
    #   validation period. This is the clearest visual diagnostic:
    #   - Good: predicted line tracks actual line with small lag/offset
    #   - Bad:  predicted line misses wave peaks or has wrong timing
    #   - Ugly: predicted line is flat or near-zero (model has collapsed
    #           to predicting the mean — a common failure mode on sparse data)
    #
    # CONTEXT: val_dates alignment — the first validation window starts at
    #   split + SEQ_LEN because the first SEQ_LEN days of the val portion
    #   are consumed as the input context window, not predicted targets.
    val_dates = dates[
        split + SEQ_LEN : split + SEQ_LEN + len(preds_raw)
    ]

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

    # Panel 1: 1-day-ahead predictions vs actuals
    axes[0].plot(val_dates, actuals_raw[:, 0],
                 label='Actual (day+1)',
                 color='steelblue', linewidth=1.2)
    axes[0].plot(val_dates, preds_raw[:, 0],
                 label='Predicted (day+1)',
                 color='tomato', linewidth=1.2, alpha=0.8)
    axes[0].set_title(
        f'Validation: 1-day-ahead Predicted vs Actual — {selected["label"]}\n'
        f'MAE={mae:.2f} deaths/day | RMSE={rmse:.2f} | '
        f'Normalized MAE={normalized_mae:.3f}',
        fontsize=11
    )
    axes[0].set_ylabel('Deaths / day')
    axes[0].legend()

    # Panel 2: absolute error over time
    # CONTEXT: Plotting absolute error separately reveals whether errors are
    #   concentrated at wave peaks (expected) or distributed randomly
    #   (suggests the model has not learned wave dynamics).
    #   Spikes at wave peaks are acceptable. Sustained high error throughout
    #   the validation period suggests the model needs more capacity or data.
    abs_error = np.abs(preds_raw[:, 0] - actuals_raw[:, 0])
    axes[1].fill_between(val_dates, abs_error,
                         color='orange', alpha=0.5,
                         label='Absolute error (day+1)')
    axes[1].axhline(y=mae, color='red', linestyle='--',
                    linewidth=1, label=f'MAE={mae:.2f}')
    axes[1].set_ylabel('Absolute error (deaths/day)')
    axes[1].legend()

    for ax in axes:
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
        ax.set_ylim(bottom=0)

    plt.tight_layout()
    plt.show()

else:
    print('Validation set was empty — no evaluation possible.')
    print('See Cell 7 warning. Choose a location with more data or reduce '
          'SEQ_LEN / FORECAST_LEN.')


## 14. Save & Reload

In [ ]:
# ── SAVE & RELOAD ─────────────────────────────────────────────────────────────
#
# CONTEXT: Three objects must be saved together to fully reconstruct the
#   forecast pipeline in a future session. Saving any subset will produce
#   a pipeline that either fails to load or produces silently wrong outputs:
#
#   1. Model weights       (covid_lstm_jhu.pt)
#      — the learned LSTM parameters. PyTorch state_dict format: a dict
#        mapping layer names to tensors. Does NOT include model architecture
#        — you must reconstruct LSTMForecaster() with identical arguments
#        before loading weights. See reload template below.
#
#   2. Scaler              (covid_scaler_jhu.pkl)
#      — the MinMaxScaler fit on this location's smoothed series in Cell 6.
#        If you reload the model without the original scaler and fit a new
#        one on different data, the model receives differently-scaled inputs
#        and produces wrong outputs with no error message.
#        CONTEXT: scaler is location-specific — it was fit on the selected
#        location's full series. It cannot be reused for a different location
#        without refitting, which requires retraining the model.
#
#   3. Metadata            (covid_metadata_jhu.pkl)
#      — a dict capturing all configuration and context needed to reconstruct
#        the pipeline without this notebook or this conversation:
#          selected_label    : human-readable location name
#          pick_state        : state string used in Cell 5
#          pick_county       : county string used in Cell 5
#          seq_len           : look-back window (must match model architecture)
#          forecast_len      : output horizon (must match model architecture)
#          last_known_date   : anchor date for forecast date generation
#          series_start_date : Jan 22 2020 — first date in JHU series
#          series_end_date   : last date in JHU series
#          n_days            : total days in series
#          hidden_size       : must match LSTMForecaster() at reload time
#          num_layers        : must match LSTMForecaster() at reload time
#          dropout           : must match LSTMForecaster() at reload time
#          data_file         : source filename
#          generated_by      : provenance note
#          notes             : free-text field for future annotations
#
# DECISION: Save metadata separately from weights, not embedded in weights.
#   CONTEXT: PyTorch's torch.save() can serialize arbitrary Python objects
#   alongside state_dict, but this couples metadata to the PyTorch format
#   and makes it harder to inspect without loading the file in PyTorch.
#   A separate pickle file is inspectable with standard Python and readable
#   by any future tool or AI without a PyTorch dependency.
#
# WARNING: These files will be overwritten silently if this cell is re-run
#   with a different location selected in Cell 5. If you want to preserve
#   a previous model, rename the files before re-running.
# REVISIT IF: saving models for multiple locations in the same directory —
#   add location to filename to avoid overwriting:
#   f'covid_lstm_{PICK_STATE}_{PICK_COUNTY}.pt'.replace(' ', '_')

import datetime

# ── SAVE MODEL WEIGHTS ────────────────────────────────────────────────────────
torch.save(model.state_dict(), 'covid_lstm_jhu.pt')
print('Saved: covid_lstm_jhu.pt')

# ── SAVE SCALER ───────────────────────────────────────────────────────────────
with open('covid_scaler_jhu.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('Saved: covid_scaler_jhu.pkl')

# ── SAVE METADATA ─────────────────────────────────────────────────────────────
metadata = {
    # Location
    'selected_label':     selected['label'],
    'pick_state':         PICK_STATE,
    'pick_county':        PICK_COUNTY,

    # Model architecture — must match LSTMForecaster() args at reload time
    'seq_len':            SEQ_LEN,
    'forecast_len':       FORECAST_LEN,
    'hidden_size':        128,
    'num_layers':         2,
    'dropout':            0.2,
    'input_size':         1,

    # Date context
    'last_known_date':    dates[-1].isoformat(),
    'series_start_date':  dates[0].isoformat(),
    'series_end_date':    dates[-1].isoformat(),
    'n_days':             n_days,

    # Provenance
    'data_file':          DATA_FILE,
    'generated_by':       'Claude (Anthropic), conversation March 2026',
    'saved_at':           datetime.datetime.now().isoformat(),

    # Free-text annotation field — add notes here for future reference
    'notes': (
        'LSTM trained on single location. '
        'Forecast horizon (90d) > look-back window (60d) — aggressive. '
        'Smoothed with 7-day centered rolling average before scaling. '
        'Negative diffs (JHU corrections) clipped to 0. '
        'See notebook comments for full decision log.'
    )
}

with open('covid_metadata_jhu.pkl', 'wb') as f:
    pickle.dump(metadata, f)
print('Saved: covid_metadata_jhu.pkl')

print('\nMetadata summary:')
for k, v in metadata.items():
    print(f'  {k:25s}: {v}')

# ── RELOAD TEMPLATE ───────────────────────────────────────────────────────────
# CONTEXT: Copy and run this block in a new session to fully reconstruct
#   the forecast pipeline without rerunning the full notebook.
#   All imports from Cell 2 must be available.
#   The original data file is NOT required for inference — only the
#   three saved files above and a raw_deaths array for the selected location.
#
# NOTE TO SELF: if reloading in a future session where the data file is
#   available, run Cells 3-5 to reconstruct raw_deaths for the saved
#   location, then use the reload template below for the pipeline.
#   If the data file is NOT available, you must supply raw_deaths from
#   another source — the saved files do not contain the time series data.

print('\n── RELOAD TEMPLATE (copy to new session) ──────────────────────────')
print("""
import pickle, torch
import numpy as np
import pandas as pd

# Load metadata first — it tells you everything needed for reconstruction
with open('covid_metadata_jhu.pkl', 'rb') as f:
    meta = pickle.load(f)
print(meta)

# Reconstruct model with IDENTICAL architecture arguments
model = LSTMForecaster(
    input_size  = meta['input_size'],
    hidden_size = meta['hidden_size'],
    num_layers  = meta['num_layers'],
    dropout     = meta['dropout'],
    forecast_len= meta['forecast_len']
)
model.load_state_dict(torch.load('covid_lstm_jhu.pt', map_location='cpu'))

# Load scaler — must be the original scaler, not a new one
with open('covid_scaler_jhu.pkl', 'rb') as f:
    scaler = pickle.load(f)

# Reconstruct pipeline
pipe = COVIDForecastPipeline(
    model          = model,
    scaler         = scaler,
    seq_len        = meta['seq_len'],
    last_known_date= meta['last_known_date']
)

# Run inference — supply raw_deaths for the saved location
result = pipe(raw_deaths)
print(result['dates'][:5], result['forecast'][:5])
""")
